In [ ]:
pip install pandas numpy rank_bm25 sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine


PHASE 1: DATA PREPARATION & SIMULATION

In [ ]:
job_description = """
Senior AI Engineer Founding Team. Need deep technical depth in modern ML systems
embeddings, retrieval, ranking, LLMs, fine-tuning. Scrappy product engineering attitude,
willing to ship. Experience designing evaluation frameworks for ranking systems like NDCG, MRR, MAP.
No pure researchers, no recent LangChain framework enthusiasts, no pure consulting firm backgrounds.
Location Pune Noida preferred.
"""

candidates_data = [
    {
        "id": 1,
        "name": "Arjun - The Hidden Gem",
        "resume": "Backend Engineer at a scaling e-commerce product startup. Built and shipped a personalized recommendation engine and high-scale information retrieval pipeline using vector similarity and custom scoring matrix. Handled embedding drift and database optimization.",
        "signals": {
            "last_active_date": "2026-06-28",
            "recruiter_response_rate": 0.90,
            "notice_period_days": 15,
            "preferred_work_mode": "hybrid",
            "willing_to_relocate": True,
            "total_experience_years": 6,
            "average_tenure_years": 3.0,
            "is_consulting_firm": False
        },
        "ground_truth_rel": 3
    },
    {
        "id": 2,
        "name": "Rahul - The Keyword Spammer (The Trap)",
        "resume": "AI Engineer. Experienced in RAG, LangChain, Pinecone, OpenAI, LangChain framework, LangChain developer, RAG expert, Pinecone vector DB.",
        "signals": {
            "last_active_date": "2026-06-25",
            "recruiter_response_rate": 0.85,
            "notice_period_days": 30,
            "preferred_work_mode": "remote",
            "willing_to_relocate": False,
            "total_experience_years": 1.2,
            "average_tenure_years": 1.2,
            "is_consulting_firm": False
        },
        "ground_truth_rel": 1
    },
    {
        "id": 3,
        "name": "Priya - The Unreachable Star",
        "resume": "Senior ML Engineer at a top product company. Designed semantic search platforms, learning-to-rank systems, and hybrid retrieval matching models.",
        "signals": {
            "last_active_date": "2025-12-10",
            "recruiter_response_rate": 0.05,
            "notice_period_days": 90,
            "preferred_work_mode": "onsite",
            "willing_to_relocate": False,
            "total_experience_years": 7,
            "average_tenure_years": 3.5,
            "is_consulting_firm": False
        },
        "ground_truth_rel": 0
    },
    {
        "id": 4,
        "name": "Amit - The Title Chaser",
        "resume": "Senior AI Architect. Expert in NLP, information retrieval, and building search systems.",
        "signals": {
            "last_active_date": "2026-06-29",
            "recruiter_response_rate": 0.95,
            "notice_period_days": 30,
            "preferred_work_mode": "hybrid",
            "willing_to_relocate": True,
            "total_experience_years": 5,
            "average_tenure_years": 0.8,
            "is_consulting_firm": False
        },
        "ground_truth_rel": 1
    }
]
print(job_description)
print(candidates_data)


Senior AI Engineer Founding Team. Need deep technical depth in modern ML systems
embeddings, retrieval, ranking, LLMs, fine-tuning. Scrappy product engineering attitude,
willing to ship. Experience designing evaluation frameworks for ranking systems like NDCG, MRR, MAP.
No pure researchers, no recent LangChain framework enthusiasts, no pure consulting firm backgrounds.
Location Pune Noida preferred.

[{'id': 1, 'name': 'Arjun - The Hidden Gem', 'resume': 'Backend Engineer at a scaling e-commerce product startup. Built and shipped a personalized recommendation engine and high-scale information retrieval pipeline using vector similarity and custom scoring matrix. Handled embedding drift and database optimization.', 'signals': {'last_active_date': '2026-06-28', 'recruiter_response_rate': 0.9, 'notice_period_days': 15, 'preferred_work_mode': 'hybrid', 'willing_to_relocate': True, 'total_experience_years': 6, 'average_tenure_years': 3.0, 'is_consulting_firm': False}, 'ground_truth_rel': 3}

In [ ]:
def preprocess_text(text):
  return text.lower().replace(".", "").replace(",", "").split()


PHASE 2: HYBRID RETRIEVAL LAYER


In [ ]:
print("Running Hybrid Retrieval Path")

Running Hybrid Retrieval Path


In [ ]:
#1.Lexical Path(BM25)
corpus=[preprocess_text(c['resume']) for c in candidates_data]
print(corpus)
bm25=BM25Okapi(corpus)
print(bm25)
query_tokens=preprocess_text(job_description)
print(query_tokens)
bm25_scores=bm25.get_scores(query_tokens)
print(bm25_scores)

[['backend', 'engineer', 'at', 'a', 'scaling', 'e-commerce', 'product', 'startup', 'built', 'and', 'shipped', 'a', 'personalized', 'recommendation', 'engine', 'and', 'high-scale', 'information', 'retrieval', 'pipeline', 'using', 'vector', 'similarity', 'and', 'custom', 'scoring', 'matrix', 'handled', 'embedding', 'drift', 'and', 'database', 'optimization'], ['ai', 'engineer', 'experienced', 'in', 'rag', 'langchain', 'pinecone', 'openai', 'langchain', 'framework', 'langchain', 'developer', 'rag', 'expert', 'pinecone', 'vector', 'db'], ['senior', 'ml', 'engineer', 'at', 'a', 'top', 'product', 'company', 'designed', 'semantic', 'search', 'platforms', 'learning-to-rank', 'systems', 'and', 'hybrid', 'retrieval', 'matching', 'models'], ['senior', 'ai', 'architect', 'expert', 'in', 'nlp', 'information', 'retrieval', 'and', 'building', 'search', 'systems']]
['senior', 'ai', 'engineer', 'founding', 'team', 'need', 'deep', 'technical', 'depth', 'in', 'modern', 'ml', 'systems', 'embeddings', 'ret

In [ ]:
#2. Dense Path(Embeddings)
embedding_path=SentenceTransformer("all-MiniLM-L6-v2")
jd_vector=embedding_path.encode(job_description)
semantic_scores=[1-cosine(jd_vector,embedding_path.encode(c['resume'])) for c in candidates_data]
candidate_vectors=[embedding_path.encode(c['resume']) for c in candidates_data]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#3. Combine and Normalize Scores (Min-Max Scaling)
def normalize(scores):
    s_min, s_max = min(scores), max(scores)
    if s_max == s_min:
        return [1.0 for _ in scores]
    return [(s - s_min) / (s_max - s_min) for s in scores]
norm_bm25 = normalize(bm25_scores)
norm_semantic = normalize(semantic_scores)
print(norm_bm25)
print(norm_semantic)

[np.float64(0.02085003719884715), np.float64(1.0), np.float64(0.4208285969601159), np.float64(0.0)]
[np.float32(0.0), np.float32(1.0), np.float32(0.8107552), np.float32(0.58206403)]


In [ ]:
hybrid_scores = [0.3 * b + 0.7 * s for b, s in zip(norm_bm25, norm_semantic)]
print(hybrid_scores)

[np.float64(0.0062550111596541445), np.float64(0.9999999880790711), np.float64(0.6937771845491554), np.float64(0.4074448049068451)]


PHASE 3: ANTI-TRAP AUDITING & SIGNAL MODIFICATION

In [ ]:
print("--- Applying Behavioral Auditing & Filters ---")

--- Applying Behavioral Auditing & Filters ---


In [ ]:
final_scored_candidates = []
current_date = datetime.strptime("2026-06-30", "%Y-%m-%d")

In [ ]:
for idx, c in enumerate(candidates_data):
    base_score = hybrid_scores[idx]
    sig = c["signals"]

In [ ]:
for c in candidates_data:
  buzzword_count = c["resume"].lower().count("langchain") + c["resume"].lower().count("rag")
  if buzzword_count > 2 and sig["total_experience_years"] < 2:
     base_score *= 0.5

In [ ]:
last_active = datetime.strptime(sig["last_active_date"], "%Y-%m-%d")
days_inactive = (current_date - last_active).days
if days_inactive > 90 or sig["recruiter_response_rate"] < 0.10:
  base_score *= 0.1

In [ ]:
if sig["notice_period_days"] <= 30:
        base_score *= 1.15
final_scored_candidates.append({
      "id": c["id"],
      "name": c["name"],
      "final_score": base_score,
      "rel": c["ground_truth_rel"]
})

In [ ]:
ranked_pipeline = sorted(final_scored_candidates, key=lambda x: x["final_score"], reverse=True)
print("\nFinal Ranked Search Results:")
for rank, cand in enumerate(ranked_pipeline, 1):
    print(f"{rank}. {cand['name']} | Adjusted Score: {cand['final_score']:.3f} | Ground Truth Relevance: {cand['rel']}")


Final Ranked Search Results:
1. Amit - The Title Chaser | Adjusted Score: 0.469 | Ground Truth Relevance: 1


PHASE 4: EVALUATION INFRASTRUCTURE

In [ ]:
print("\n--- Running System Evaluation Analytics ---")


--- Running System Evaluation Analytics ---


In [ ]:
def calculate_mrr(ranked_results):
    for i, res in enumerate(ranked_results, 1):
        # Extract ground truth relevance safely
        rel = res["rel"] if isinstance(res, dict) else res
        if int(rel) == 3:  # Target top-tier candidate
            return 1.0 / i
    return 0.0

In [ ]:
def calculate_ndcg(ranked_results, k=4):
    dcg = 0.0
    for i, res in enumerate(ranked_results[:k], 1):
        rel = res["rel"] if isinstance(res, dict) else res
        dcg += int(rel) / np.log2(i + 1)
    ideal_results = sorted(ranked_results, key=lambda x: int(x["rel"]) if isinstance(x, dict) else int(x), reverse=True)
    idcg = 0.0
    for i, res in enumerate(ideal_results[:k], 1):
        rel = res["rel"] if isinstance(res, dict) else res
        idcg += int(rel) / np.log2(i + 1)

    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
mrr_score = calculate_mrr(ranked_pipeline)
ndcg_score = calculate_ndcg(ranked_pipeline, k=4)

print(f"Engine MRR Metric: {mrr_score:.4f}")
print(f"Engine NDCG@4 Metric: {ndcg_score:.4f}")

Engine MRR Metric: 0.0000
Engine NDCG@4 Metric: 1.0000
